# 11 - Integrador conceptual: copiloto GitHub con MCP

Objetivo: unir discovery, tool, resource, prompt, schema, transporte y audit antes de abrir el Python real.


## Grafico guia

```mermaid
flowchart LR
    A["Antigravity o Host"] --> B["MCP Client"]
    B --> C["GitHub MCP Server"]
    C --> D["github_create_repo"]
    C --> E["github_upsert_file"]
    C --> F["github://config"]
    D --> G["Repo privado"]
    E --> G
```


In [ ]:
from pydantic import BaseModel, Field, ValidationError
from uuid import uuid4

class CreateRepoInput(BaseModel):
    name: str = Field(..., min_length=3)
    description: str = ""
    private: bool = True

class GitHubConceptMCPServer:
    def __init__(self):
        self.resources = {"github://config": {"token_configured": True, "default_private_repos": True}}
        self.prompts = {"repo_bootstrap_prompt": "Create private repo {project_name} for {goal}."}
        self.audit = []

    def list_capabilities(self):
        return {
            "resources": list(self.resources),
            "prompts": list(self.prompts),
            "tools": ["github_create_repo", "github_upsert_file", "github_get_repo"],
        }

    def read_resource(self, uri):
        return self.resources.get(uri, {"error": "resource_not_found"})

    def get_prompt(self, name):
        return self.prompts.get(name, {"error": "prompt_not_found"})


In [ ]:
def create_audit(server, tool, status, detail=None):
    event = {"request_id": "req_" + uuid4().hex[:8], "tool": tool, "status": status, "detail": detail or {}}
    server.audit.append(event)
    return event

def call_tool(server, name, payload):
    if name == "github_create_repo":
        try:
            parsed = CreateRepoInput.model_validate(payload)
        except ValidationError as exc:
            return {"error": "validation_error", "details": exc.errors(), "audit": create_audit(server, name, "failed")}
        create_audit(server, name, "success", {"repo": parsed.name, "private": parsed.private})
        return {"full_name": f"demo/{parsed.name}", "html_url": f"https://github.com/demo/{parsed.name}", "private": parsed.private}
    if name == "github_upsert_file":
        create_audit(server, name, "success", {"path": payload.get("path"), "content_chars": len(payload.get("content", ""))})
        return {"path": payload.get("path"), "commit_sha": "abc123"}
    return {"error": "unknown_tool", "audit": create_audit(server, name, "failed")}


In [ ]:
server = GitHubConceptMCPServer()
capabilities = server.list_capabilities()
config = server.read_resource("github://config")
prompt = server.get_prompt("repo_bootstrap_prompt")
repo = call_tool(server, "github_create_repo", {"name": "aem4l3-mcp-demo", "description": "demo", "private": True})
readme = call_tool(server, "github_upsert_file", {"path": "README.md", "content": "# Demo MCP"})
capabilities, config, prompt, repo, readme, server.audit


In [ ]:
error_cases = {
    "invalid_repo_name": call_tool(server, "github_create_repo", {"name": "x"}),
    "missing_resource": server.read_resource("github://missing"),
    "unknown_tool": call_tool(server, "github_delete_repo", {}),
}
error_cases


## Cierre docente

Este notebook une todo: discovery, resource, prompt, tool, schema, audit y manejo de errores. Despues de esto, abrir `e01_github_mcp_server.py` y mostrar que la misma arquitectura ahora toca GitHub real.
